In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpwxqssuc3".


In [ ]:
%%cuda

#include <stdio.h>

__global__ void helloCUDA() {
    printf("Hello from GPU thread %d!\n", threadIdx.x);
}

int main() {
    helloCUDA<<<2, 5>>>();
    cudaDeviceSynchronize();
    return 0;
}

Hello from GPU thread 0!
Hello from GPU thread 1!
Hello from GPU thread 2!
Hello from GPU thread 3!
Hello from GPU thread 4!
Hello from GPU thread 0!
Hello from GPU thread 1!
Hello from GPU thread 2!
Hello from GPU thread 3!
Hello from GPU thread 4!



In [ ]:
%%cuda
#include <stdio.h>
#include <cuda_runtime.h>

#define N 1024   // Size of vectors


__global__ void vectorAdd(float *A, float *B, float *C, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    float *h_A, *h_B, *h_C;     // Host vectors
    float *d_A, *d_B, *d_C;     // Device vectors
    int size = N * sizeof(float);

    // Allocate memory on Host (CPU)
    h_A = (float*)malloc(size);
    h_B = (float*)malloc(size);
    h_C = (float*)malloc(size);

    // Initialize vectors
    for(int i = 0; i < N; i++)
    {
        h_A[i] = i;
        h_B[i] = i * 2;
    }

    // Allocate memory on Device (GPU)
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // Copy data from Host to Device
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Kernel configuration
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    // Launch kernel
    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    // Copy result back to Host
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // Print some results
    printf("Vector Addition Result:\n");
    for(int i = 0; i < 10; i++)
    {
        printf("%f + %f = %f\n", h_A[i], h_B[i], h_C[i]);
    }

    // Free device memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    // Free host memory
    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Vector Addition Result:
0.000000 + 0.000000 = 0.000000
1.000000 + 2.000000 = 3.000000
2.000000 + 4.000000 = 6.000000
3.000000 + 6.000000 = 9.000000
4.000000 + 8.000000 = 12.000000
5.000000 + 10.000000 = 15.000000
6.000000 + 12.000000 = 18.000000
7.000000 + 14.000000 = 21.000000
8.000000 + 16.000000 = 24.000000
9.000000 + 18.000000 = 27.000000



# just try

In [ ]:
%%cuda
#include<stdio.h>
#define N 5

__global__ void add(int * arr,int a){
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if(i<10)
      arr[i] = arr[i] * a;
}



int main(){
    srand(time(NULL));
    int size;
    int arr[10];
    size = 10*sizeof(int);
    for(int i =0 ;i<10;i++){
        arr[i] = rand()%10;
        printf("%d ",arr[i]);
    }
    int *d_arr;
    cudaMalloc((void **)&d_arr,size);
    cudaMemcpy(d_arr,arr,size,cudaMemcpyHostToDevice);
    int n;
    add<<<(n+31)/32,32>>>(d_arr,n);
    cudaDeviceSynchronize();
    cudaMemcpy(arr,d_arr,size,cudaMemcpyDeviceToHost);
    printf("\n");
    for(int i =0 ;i<10;i++){
        printf("%d ",arr[i]);
    }
    cudaFree(d_arr);
           return 0;
}

9 9 9 0 6 7 3 1 4 1 
90 90 90 0 60 70 30 10 40 10 


1.Write a CUDA program where each GPU thread prints "I am thread X in block Y".

In [ ]:


%%cuda
#include<stdio.h>
#include<cuda_runtime.h>

__global__ void fun(){
    printf("i am thread %d my global thread id is %d in block %d!\n",threadIdx.x,blockDim.x*blockIdx.x+threadIdx.x,blockIdx.x);
}

int main(){
    fun<<<3,4>>>();
    cudaDeviceSynchronize();
    return 0;
}

i am thread 0 my global thread id is 8 in block 2!
i am thread 1 my global thread id is 9 in block 2!
i am thread 2 my global thread id is 10 in block 2!
i am thread 3 my global thread id is 11 in block 2!
i am thread 0 my global thread id is 0 in block 0!
i am thread 1 my global thread id is 1 in block 0!
i am thread 2 my global thread id is 2 in block 0!
i am thread 3 my global thread id is 3 in block 0!
i am thread 0 my global thread id is 4 in block 1!
i am thread 1 my global thread id is 5 in block 1!
i am thread 2 my global thread id is 6 in block 1!
i am thread 3 my global thread id is 7 in block 1!



2. Write a CUDA program to add two vectors A and B of size N.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<cuda_runtime.h>
#include<time.h>
#define N 5


__global__ void add_two_vec(int *d_arr_A,int *d_arr_B,int *d_arr_C){
    int i = blockIdx.x*blockDim.x+threadIdx.x;
    if(i < N)
    d_arr_C[i] = d_arr_A[i] + d_arr_B[i];
}


int main(){
    int *h_arr_A;
    int *h_arr_B;
    int *h_arr_C;

    int *d_arr_A;
    int *d_arr_B;
    int *d_arr_C;

    srand(time(NULL));
    h_arr_A = (int *)malloc(N*sizeof(int));
    h_arr_B = (int *)malloc(N*sizeof(int));
    h_arr_C = (int *)malloc(N*sizeof(int));

    cudaMalloc((void **)&d_arr_A,N*sizeof(int));
    cudaMalloc((void **)&d_arr_B,N*sizeof(int));
    cudaMalloc((void **)&d_arr_C,N*sizeof(int));

    printf("The Vetcor A is:\n");
    for(int i = 0;i<N;i++){
        h_arr_A[i] = rand()%100;
        printf("%d ",h_arr_A[i]);
    }
    printf("\nThe Vetcor B is:\n");
    for(int i = 0;i<N;i++){
        h_arr_B[i] = rand()%100;
        printf("%d ",h_arr_B[i]);
    }

    cudaMemcpy(d_arr_A,h_arr_A,N*sizeof(int),cudaMemcpyHostToDevice);
    cudaMemcpy(d_arr_B,h_arr_B,N*sizeof(int),cudaMemcpyHostToDevice);

    add_two_vec<<<1,5>>>(d_arr_A,d_arr_B,d_arr_C);
    cudaDeviceSynchronize();

    cudaMemcpy(h_arr_C,d_arr_C,N*sizeof(int),cudaMemcpyDeviceToHost);

    printf("\nThe resultant Vector C is:\n");
    for(int i = 0;i<N;i++){
        printf("%d ",h_arr_C[i]);
    }

    cudaFree(d_arr_A);
    cudaFree(d_arr_B);
    cudaFree(d_arr_C);

    free(h_arr_A);
    free(h_arr_B);
    free(h_arr_C);
    return 0;
}

The Vetcor A is:
17 28 98 19 28 
The Vetcor B is:
77 16 45 48 57 
The resultant Vector C is:
94 44 143 67 85 


3.Write a CUDA program where each thread multiplies one vector element by a constant scalar.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<cuda_runtime.h>
#include<time.h>
#define N 10


__global__ void vector_mul(int *arr,int con){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if(i<N)
    arr[i] = arr[i] * con;
}

int main(){
    int size = N*sizeof(int);
    srand(time(NULL));

    int *h_arr;
    h_arr = (int *)malloc(size);
    int *d_arr;
    cudaMalloc((void **)&d_arr,size);
    int h_con; // using normal variable

    printf("the vector is:\n");
    for(int i = 0;i<N;i++){
        h_arr[i] = rand()%20;
        printf("%d ",h_arr[i]);
    }
    h_con = rand()%20;
    printf("\nthe constant is: %d",h_con);

    cudaMemcpy(d_arr,h_arr,size,cudaMemcpyHostToDevice);

    vector_mul<<<2,5>>>(d_arr,h_con);
    cudaDeviceSynchronize();
    cudaMemcpy(h_arr,d_arr,size,cudaMemcpyDeviceToHost);

    printf("\nthe vector is:\n");
    for(int i = 0;i<N;i++){
        printf("%d ",h_arr[i]);
    }
    cudaFree(d_arr);
    free(h_arr);

    return 0;
}

the vector is:
15 6 10 19 16 4 19 10 0 12 
the constant is: 6
the vector is:
90 36 60 114 96 24 114 60 0 72 


4.Write a CUDA kernel where each thread checks whether its assigned number is even or odd
and store results in another array.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<cuda_runtime.h>
#include<time.h>
#define N 10

__global__ void evv_odd(int * arr_EvOd,int * res){
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i<N)
    res[i] = arr_EvOd[i]%2;
}
int main(){
    int size = N*sizeof(int);
    int *h_arr;
    int *d_arr;
    int *h_res;
    int *d_res;
    srand(time(NULL));

    h_arr = (int *)malloc(size);
    h_res = (int *)malloc(size);

    cudaMalloc((void **)&d_arr,size);
    cudaMalloc((void **)&d_res,size);


    printf("the array is:\n");
    for(int i = 0;i<N;i++){
        h_arr[i] = rand()%100;
        printf("%d ",h_arr[i]);
    }
    cudaMemcpy(d_arr,h_arr,size,cudaMemcpyHostToDevice);

    evv_odd<<<2,5>>>(d_arr,d_res);
    cudaDeviceSynchronize();

    cudaMemcpy(h_res,d_res,size,cudaMemcpyDeviceToHost);
    printf("\nThe Result array: \n");
    for(int i = 0;i<N;i++){
        printf("%d is ",h_arr[i]);
        if(h_res[i] == 0){
            printf("even");
        }
        else{
            printf("odd");
        }
        printf("\n");
    }
    cudaFree(d_arr);
    cudaFree(d_res);
    free(h_arr);
    free(h_res);

    return 0;
}

the array is:
36 26 1 45 71 18 63 15 97 90 
The Result array: 
36 is even
26 is even
1 is odd
45 is odd
71 is odd
18 is even
63 is odd
15 is odd
97 is odd
90 is even



5.Write a CUDA program to reverse an array in parallel.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<time.h>
#include<cuda_runtime.h>
#define N 10



__global__ void reverse(int * arr,int * rev){
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < N)
    rev[i] = arr[N-i-1];
}

int main(){
    int *h_arr,*h_rev;
    int *d_arr,*d_rev;
    int size = N*sizeof(int);
    srand(time(NULL));

    h_arr = (int *)malloc(size);
    h_rev = (int *)malloc(size);
    cudaMalloc((void **)&d_arr,size);
    cudaMalloc((void **)&d_rev,size);

    printf("The Original array is:\n");
    for (int i = 0;i<N;i++){
        h_arr[i] = rand()%100;
        printf("%d ",h_arr[i]);
    }

    cudaMemcpy(d_arr,h_arr,size,cudaMemcpyHostToDevice);

    reverse<<<2,N/2>>>(d_arr,d_rev);

    cudaMemcpy(h_rev,d_rev,size,cudaMemcpyDeviceToHost);
    cudaDeviceSynchronize();

    printf("\nAfter reversing the original array is:\n");
    for (int i = 0;i<N;i++){
        printf("%d ",h_rev[i]);
    }

    cudaFree(d_arr);
    cudaFree(d_rev);
    free(h_arr);
    free(h_rev);

    return 0;
}

The Original array is:
61 81 22 34 88 50 15 98 89 87 
After reversing the original array is:
87 89 98 15 50 88 34 22 81 61 


6.Write a CUDA program to search a key element in an array.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<cuda_runtime.h>
#include<time.h>
#define N 10



__global__ void search(int *arr,int key,int *is_found){
    int i = blockDim.x*blockIdx.x+threadIdx.x;
    if(i<N){
        if (arr[i] == key){
            atomicExch(is_found,1);
        }
    }
}

int main(){
    int size = N*sizeof(int);
    int *h_arr,*h_is_found,key;
    int *d_arr,*d_is_found;
    srand(time(NULL));

    h_arr = (int *)malloc(size);
    h_is_found = (int *)malloc(sizeof(int));

    cudaMalloc((void **)&d_arr,size);
    cudaMalloc((void **)&d_is_found,sizeof(int));
    printf("The Array is:\n");
    for(int i = 0;i<N;i++){
        h_arr[i] = rand()%15;
        // h_arr[i] = i+10;
        printf("%d ",h_arr[i]);
    }
    printf("\n");
    key = 5;
    printf("The key is: %d\n",key);
    *h_is_found = 0;

    cudaMemcpy(d_arr,h_arr,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_is_found,h_is_found,sizeof(int),cudaMemcpyHostToDevice);

    search<<<3,5>>>(d_arr,key,d_is_found);
    cudaDeviceSynchronize();

    cudaMemcpy(h_is_found,d_is_found,sizeof(int),cudaMemcpyDeviceToHost);
    if(*h_is_found){
        printf("key is found");
    }
    else{
        printf("key is not found");
    }

    cudaFree(d_arr);
    cudaFree(d_is_found);
    free(h_arr);
    free(h_is_found);

    return 0;
}

The Array is:
14 10 10 11 1 1 5 12 7 7 
The key is: 5
key is found


7.Write a CUDA program where each thread processes part of the array and find the maximum
element using parallel reduction.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<time.h>

#define N 8   // better power of 2

__global__ void maximum(int *arr){
    int tid = threadIdx.x;

    for(int stride = blockDim.x/2; stride > 0; stride /= 2){
        if(tid < stride){
            if(arr[tid] < arr[tid + stride]){
                arr[tid] = arr[tid + stride];
            }
        }
        __syncthreads();
    }
}

int main(){
    int *h_arr;
    int *d_arr;
    int size = N * sizeof(int);

    srand(time(NULL));

    h_arr = (int*)malloc(size);
    cudaMalloc((void**)&d_arr, size);

    printf("The Original array:\n");
    for(int i = 0; i < N; i++){
        h_arr[i] = rand() % 50;
        printf("%d ", h_arr[i]);
    }
    printf("\n");

    cudaMemcpy(d_arr, h_arr, size, cudaMemcpyHostToDevice);

    maximum<<<1, N>>>(d_arr);

    cudaMemcpy(h_arr, d_arr, size, cudaMemcpyDeviceToHost);

    printf("The Maximum element in this array: %d\n", h_arr[0]);

    cudaFree(d_arr);
    free(h_arr);

    return 0;
}

The Original array:
12 37 36 24 17 33 2 15 
The Maximum element in this array: 37



8. Write a CUDA program to compute dot product of two vectors.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<cuda_runtime.h>
#include<time.h>
#define N 3


__global__ void dot_prod(int *arr1,int *arr2,int *sum){
    int i = blockDim.x*blockIdx.x+threadIdx.x;
    if(i<N)
    atomicAdd(sum,arr1[i]*arr2[i]);
}

int main(){
    int size = N*sizeof(int);
    int *h_arr_A,*h_arr_B,*h_sum;
    int *d_arr_A,*d_arr_B,*d_sum;
    srand(time(NULL));

    h_arr_A = (int *)malloc(size);
    h_arr_B = (int *)malloc(size);
    h_sum = (int *)malloc(sizeof(int));
    *h_sum = 0;

    cudaMalloc((void **)&d_arr_A,size);
    cudaMalloc((void **)&d_arr_B,size);
    cudaMalloc((void **)&d_sum,sizeof(int));

    for(int i = 0;i<N;i++){
            h_arr_A[i] = rand()%100;
            h_arr_B[i] = rand()%100;
    }

    printf("The Vector A:\n");
    for(int i = 0;i<N;i++){
            printf("%d ",h_arr_A[i]);
    }
        printf("\n");
    printf("The Vector B:\n");
    for(int i = 0;i<N;i++){
            printf("%d ",h_arr_B[i]);
    }

    cudaMemcpy(d_arr_A,h_arr_A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_arr_B,h_arr_B,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_sum,h_sum,size,cudaMemcpyHostToDevice);

    dot_prod<<<1,N>>>(d_arr_A,d_arr_B,d_sum);
    cudaDeviceSynchronize();

    cudaMemcpy(h_sum,d_sum,sizeof(int),cudaMemcpyDeviceToHost);

    printf("\nThe result of dot product of two vectors is: %d",*h_sum);

    cudaFree(d_arr_A);
    cudaFree(d_arr_B);
    cudaFree(d_sum);
    free(h_arr_A);
    free(h_arr_B);
    free(h_sum);


    return 0;

}

The Vector A:
1 22 88 
The Vector B:
37 74 80 
The result of dot product of two vectors is: 8705


9. Given an array of numbers from 0–50, compute the frequency of each number using CUDA
threads.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<cuda_runtime.h>
#include<time.h>
#define N 50
#define range 51




__global__ void freq(int *arr,int *freq_arr){
    int i = blockDim.x*blockIdx.x+threadIdx.x;
    if (i<N){
        int temp = arr[i];
        atomicAdd(&freq_arr[temp],1);
    }
}

int main(){
    int size = N*sizeof(int);
    int h_arr[N],h_freq_arr[range]={0};
    int *d_arr,*d_freq_arr;
    srand(time(NULL));

    cudaMalloc((void **)&d_arr,size);
    cudaMalloc((void **)&d_freq_arr,range*sizeof(int));

    printf("The vector is:\n");
    for(int i = 0;i<N;i++){
        h_arr[i]=rand()%51;
        printf("%d ",h_arr[i]);
    }

    cudaMemcpy(d_arr,h_arr,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_freq_arr,h_freq_arr,range*sizeof(int),cudaMemcpyHostToDevice);

    freq<<<1,N>>>(d_arr,d_freq_arr);
    cudaDeviceSynchronize();

    cudaMemcpy(h_freq_arr,d_freq_arr,range*sizeof(int),cudaMemcpyDeviceToHost);

    printf("\n");
    for(int i = 0;i<range;i++){
        if(h_freq_arr[i]!=0)
        printf("%d occured %d times\n",i,h_freq_arr[i]);
    }

    cudaFree(d_arr);
    cudaFree(d_freq_arr);
}

The vector is:
26 5 40 2 41 28 39 41 33 43 39 37 46 20 27 27 20 40 37 41 47 35 20 33 46 19 6 7 4 22 4 30 2 44 6 43 46 45 7 3 12 47 40 7 16 41 8 37 30 45 
2 occured 2 times
3 occured 1 times
4 occured 2 times
5 occured 1 times
6 occured 2 times
7 occured 3 times
8 occured 1 times
12 occured 1 times
16 occured 1 times
19 occured 1 times
20 occured 3 times
22 occured 1 times
26 occured 1 times
27 occured 2 times
28 occured 1 times
30 occured 2 times
33 occured 2 times
35 occured 1 times
37 occured 3 times
39 occured 2 times
40 occured 3 times
41 occured 4 times
43 occured 2 times
44 occured 1 times
45 occured 2 times
46 occured 3 times
47 occured 2 times



10. Write a CUDA program to add two matrices.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<cuda_runtime.h>
#include<time.h>
#define N 3



__global__ void add(int *arr1,int *arr2,int *arr3){
    int i = blockDim.x*blockIdx.x+threadIdx.x;
    if(i<N*N)
    arr3[i] = arr2[i] + arr1[i];
}

int main(){
    int size = N*N*sizeof(int);
    int *h_arr_A,*h_arr_B,*h_arr_C;
    int *d_arr_A,*d_arr_B,*d_arr_C;
    srand(time(NULL));

    h_arr_A = (int *)malloc(size);
    h_arr_B = (int *)malloc(size);
    h_arr_C = (int *)malloc(size);

    cudaMalloc((void **)&d_arr_A,size);
    cudaMalloc((void **)&d_arr_B,size);
    cudaMalloc((void **)&d_arr_C,size);

    for(int i = 0;i<N;i++){
        for(int j = 0;j<N;j++){
            h_arr_A[i*N+j] = rand()%100;
            h_arr_B[i*N+j] = rand()%100;
        }
    }

    printf("The Matrix A:\n");
    for(int i = 0;i<N;i++){
        for(int j = 0;j<N;j++){
            printf("%d ",h_arr_A[i*N+j]);
        }
        printf("\n");
    }
        printf("\n");
    printf("The Matrix B:\n");
    for(int i = 0;i<N;i++){
        for(int j = 0;j<N;j++){
            printf("%d ",h_arr_B[i*N+j]);
        }
        printf("\n");
    }

    cudaMemcpy(d_arr_A,h_arr_A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_arr_B,h_arr_B,size,cudaMemcpyHostToDevice);

    add<<<N,N>>>(d_arr_A,d_arr_B,d_arr_C);
    cudaDeviceSynchronize();

    cudaMemcpy(h_arr_C,d_arr_C,size,cudaMemcpyDeviceToHost);

    printf("\nThe resultant Matrix C:\n");
    for(int i = 0;i<N;i++){
        for(int j = 0;j<N;j++){
            printf("%d ",h_arr_C[i*N+j]);
        }
        printf("\n");
    }

    cudaFree(d_arr_A);
    cudaFree(d_arr_B);
    cudaFree(d_arr_C);
    free(h_arr_A);
    free(h_arr_B);
    free(h_arr_C);

    return 0;

}

The Matrix A:
66 61 30 
74 26 36 
82 24 43 

The Matrix B:
10 11 27 
70 72 31 
81 27 26 

The resultant Matrix C:
76 72 57 
144 98 67 
163 51 69 



11. Write a CUDA program to multiply two matrices.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<time.h>
#define row_1 3
#define col_1 5
#define row_2 5
#define col_2 3

__global__ void mult(int * arr1,int * arr2, int * arr3,int i,int k,int j){
    int c = blockDim.x*blockIdx.x+threadIdx.x;
    int r = blockDim.y*blockIdx.y+threadIdx.y;
    if(r < i && c < j){
        int sum = 0;
        for(int a = 0;a<k;a++){
            sum += arr1[r*k+a] * arr2[a*j+c];
        }
        arr3[r*j+c] = sum;
    }
}

int main(){
    srand(time(NULL));
    if(col_1 == row_2){
      int arr_1[row_1][col_1];
      int size_1 = row_1 * col_1 * sizeof(int);
      int size_2 = row_2 * col_2 * sizeof(int);
      int size_3 = row_1 * col_2 * sizeof(int);
      printf("\nThe matrix A:\n");
      for(int i = 0;i<row_1;i++){
          for(int j = 0;j<col_1;j++){
              arr_1[i][j] = rand()%20;
              printf("%d ",arr_1[i][j]);
          }
          printf("\n");
      }

      int arr_2[row_2][col_2];
      printf("\nThe matrix B:\n");
      for(int i = 0;i<row_2;i++){
          for(int j = 0;j<col_2;j++){
              arr_2[i][j] = rand()%20;
              printf("%d ",arr_2[i][j]);
          }
          printf("\n");
      }
      int arr_3[row_1][col_2]={0};

      int *d_arr1,*d_arr2,*d_arr3;

      cudaMalloc((void **)&d_arr1,size_1);
      cudaMalloc((void **)&d_arr2,size_2);
      cudaMalloc((void **)&d_arr3,size_3);

      cudaMemcpy(d_arr1,arr_1,size_1,cudaMemcpyHostToDevice);
      cudaMemcpy(d_arr2,arr_2,size_2,cudaMemcpyHostToDevice);
      cudaMemcpy(d_arr3,arr_3,size_3,cudaMemcpyHostToDevice);

      dim3 threads(16,16);
      dim3 blocks((col_2+15)/16 , (row_1+15)/16);

      mult<<<blocks,threads>>>(d_arr1,d_arr2,d_arr3,row_1,col_1,col_2);
      cudaDeviceSynchronize();

      cudaMemcpy(arr_3,d_arr3,size_3,cudaMemcpyDeviceToHost);


      printf("\nThe result matrix C:\n");
      for(int i = 0;i<row_1;i++){
          for(int j = 0;j<col_2;j++){
              printf("%d ",arr_3[i][j]);
          }
          printf("\n");
      }

    }
    else{
        printf("dimention of the matrix is not multipliable\n");
    }

}


The matrix A:
15 10 16 5 3 
14 9 5 4 17 
12 6 0 16 11 

The matrix B:
1 10 5 
19 16 15 
0 17 3 
17 15 2 
3 5 8 

The result matrix C:
299 672 307 
304 514 364 
431 511 270 



12. Write a CUDA program to compute matrix transpose.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<time.h>
#define row_1 3
#define col_1 6

__global__ void mult(int * arr1,int * arr2,int i,int j){
    int c = blockDim.x*blockIdx.x+threadIdx.x;
    int r = blockDim.y*blockIdx.y+threadIdx.y;
    if(r < i && c < j){
        arr2[c * i + r] = arr1[r * j + c];
    }
}

int main(){
    srand(time(NULL));
      int arr_1[row_1][col_1];
      int arr_2[col_1][row_1];
      printf("\nThe matrix A\n");
      for(int i = 0;i<row_1;i++){
          for(int j = 0;j<col_1;j++){
              arr_1[i][j] = rand()%100;
              printf("%d ",arr_1[i][j]);
          }
          printf("\n");
      }


      int *d_arr1,*d_arr2;

      int size_1 = row_1 * col_1 * sizeof(int);
      cudaMalloc((void **)&d_arr1,size_1);
      cudaMalloc((void **)&d_arr2,size_1);

      cudaMemcpy(d_arr1,arr_1,size_1,cudaMemcpyHostToDevice);

      dim3 threads(16,16);
      dim3 blocks((col_1+15)/16 , (row_1+15)/16);

      mult<<<blocks,threads>>>(d_arr1,d_arr2,row_1,col_1);
      cudaDeviceSynchronize();

      cudaMemcpy(arr_2,d_arr2,size_1,cudaMemcpyDeviceToHost);


      printf("\nThis is transposed matrix\n");
      for(int i = 0;i<col_1;i++){
          for(int j = 0;j<row_1;j++){
              printf("%d ",arr_2[i][j]);
          }
          printf("\n");
      }

    }



The matrix A
31 76 24 36 51 89 
29 1 82 95 50 80 
66 8 7 60 82 43 

This is transposed matrix
31 29 66 
76 1 8 
24 82 7 
36 95 60 
51 50 82 
89 80 43 



13. Consider a grayscale image represented as a 2D array. Write a CUDA program where each
thread increases pixel intensity by a constant value. Ensure pixel values remain within valid
range.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<time.h>
#define row_1 5
#define col_1 5
#define cons 10

__global__ void mult(int * arr1,int * arr2,int i,int j){
    int c = blockDim.x*blockIdx.x+threadIdx.x;
    int r = blockDim.y*blockIdx.y+threadIdx.y;
    if(r < i && c < j){
        arr2[r*j+c] = min(arr1[r*j+c] + cons,255) ;
    }
}

int main(){
    srand(time(NULL));
      int arr_1[row_1][col_1];
      int arr_2[row_1][col_1];
      printf("\nThis is gray scale image\n");
      for(int i = 0;i<row_1;i++){
          for(int j = 0;j<col_1;j++){
              arr_1[i][j] = rand()%256;
              printf("%3d ",arr_1[i][j]);
          }
          printf("\n");
      }


      int *d_arr1,*d_arr2;

      int size_1 = row_1 * col_1 * sizeof(int);
      cudaMalloc((void **)&d_arr1,size_1);
      cudaMalloc((void **)&d_arr2,size_1);

      cudaMemcpy(d_arr1,arr_1,size_1,cudaMemcpyHostToDevice);

      dim3 threads(16,16);
      dim3 blocks((col_1+15)/16 , (row_1+15)/16);

      mult<<<blocks,threads>>>(d_arr1,d_arr2,row_1,col_1);
      cudaDeviceSynchronize();

      cudaMemcpy(arr_2,d_arr2,size_1,cudaMemcpyDeviceToHost);


      printf("\nThis is after increased intensity of each pixel\n");
      for(int i = 0;i<row_1;i++){
          for(int j = 0;j<col_1;j++){
              printf("%3d ",arr_2[i][j]);
          }
          printf("\n");
      }

    }


This is gray scale image
  1 215 126 194  30 
163 233 247 147  65 
  8 115 167 157 160 
183  75  47 255 171 
 15 241  59 184  37 

This is after increased intensity of each pixel
 11 225 136 204  40 
173 243 255 157  75 
 18 125 177 167 170 
193  85  57 255 181 
 25 251  69 194  47 



14. Write a CUDA program where each thread processes a pixel. If the pixel value is greater
than a threshold, then set it to 255, otherwise, set it to 0.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<time.h>
#define row_1 5
#define col_1 5
#define cons 10

__global__ void mult(int * arr1,int * arr2,int i,int j){
    int c = blockDim.x*blockIdx.x+threadIdx.x;
    int r = blockDim.y*blockIdx.y+threadIdx.y;
    if(r < i && c < j){
        if(arr1[r*j+c] >= 255)
            arr2[r*j+c] = 255;
        else
            arr2[r*j+c] = 0;
    }
}

int main(){
    srand(time(NULL));
      int arr_1[row_1][col_1];
      int arr_2[row_1][col_1];
      printf("\nThis is gray scale image\n");
      for(int i = 0;i<row_1;i++){
          for(int j = 0;j<col_1;j++){
              arr_1[i][j] = rand()%512;
              printf("%3d ",arr_1[i][j]);
          }
          printf("\n");
      }


      int *d_arr1,*d_arr2;

      int size_1 = row_1 * col_1 * sizeof(int);
      cudaMalloc((void **)&d_arr1,size_1);
      cudaMalloc((void **)&d_arr2,size_1);

      cudaMemcpy(d_arr1,arr_1,size_1,cudaMemcpyHostToDevice);

      dim3 threads(16,16);
      dim3 blocks((col_1+15)/16 , (row_1+15)/16);

      mult<<<blocks,threads>>>(d_arr1,d_arr2,row_1,col_1);
      cudaDeviceSynchronize();

      cudaMemcpy(arr_2,d_arr2,size_1,cudaMemcpyDeviceToHost);


      printf("\nThis is after normalized each pixel\n");
      for(int i = 0;i<row_1;i++){
          for(int j = 0;j<col_1;j++){
              printf("%3d ",arr_2[i][j]);
          }
          printf("\n");
      }

    }


This is gray scale image
490 307   2 229  41 
 32  35 480  17 500 
299 136 224 142 103 
 74 201 146 497  36 
119 246  71 390 372 

This is after normalized each pixel
255 255   0   0   0 
  0   0 255   0 255 
255   0   0   0   0 
  0   0   0 255   0 
  0   0   0 255 255 



15. Write a CUDA program using 3D thread blocks to process volumetric data where each
thread processes one voxel and applies a simple operation.

In [ ]:
%%cuda
#include<stdio.h>
#include<stdlib.h>
#include<time.h>
#define row_1 3
#define col_1 3
#define brd_1 3

__global__ void mult(int *A,int *B,int rows,int cols,int depth)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    if(x < cols && y < rows && z < depth)
    {
        int index = z*(rows*cols) + y*cols + x;

        B[index] = A[index] * 2;
    }
}

int main(){
    srand(time(NULL));
      int arr_1[row_1][col_1][brd_1];
      int arr_2[col_1][row_1][brd_1];
      printf("\nThis is 3D cube:\n");
      for(int i = 0;i<row_1;i++){
          for(int j = 0;j<col_1;j++){
              for(int k = 0;k<brd_1;k++){

              arr_1[i][j][k] = rand()%10;
              printf("%d ",arr_1[i][j][k]);
              }
          printf("\n");
          }
          printf("\n");
      }


      int *d_arr1,*d_arr2;

      int size_1 = row_1 * col_1 * brd_1 * sizeof(int);
      cudaMalloc((void **)&d_arr1,size_1);
      cudaMalloc((void **)&d_arr2,size_1);

      cudaMemcpy(d_arr1,arr_1,size_1,cudaMemcpyHostToDevice);

    dim3 threads(8,8,8);
dim3 blocks(
    (col_1+7)/8,
    (row_1+7)/8,
    (brd_1+7)/8
);

      mult<<<blocks,threads>>>(d_arr1,d_arr2,row_1,col_1,brd_1);
      cudaDeviceSynchronize();

      cudaMemcpy(arr_2,d_arr2,size_1,cudaMemcpyDeviceToHost);


      printf("\nThis is the cube after simple calculation with each voxel:\n");
      for(int i = 0;i<col_1;i++){
          for(int j = 0;j<row_1;j++){
              for(int k = 0;k<brd_1;k++){
                  printf("%d ",arr_2[i][j][k]);
              }
              printf("\n");
          }
          printf("\n");
      }

    }


This is 3D cube:
2 1 7 
0 9 9 
3 0 0 

7 2 9 
7 5 3 
6 7 9 

0 1 0 
8 1 1 
0 0 6 


This is the cube after simple calculation with each voxel:
4 2 14 
0 18 18 
6 0 0 

14 4 18 
14 10 6 
12 14 18 

0 2 0 
16 2 2 
0 0 12 




In [ ]:
# Write a CUDA program where each thread processes an element of an array and replaces it with the difference between the current element and its previous element. Handle boundary conditions properly.

In [2]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp5snclew4".


In [10]:
# Write a CUDA program where each thread processes an element of an array and replaces it with the difference between the current element and its previous element. Handle boundary conditions properly.